# SparkOCR-VLM on Databricks Free Edition

Run this notebook inside a Databricks Free workspace.  
No cluster config needed — serverless compute handles everything.

**Before running:** upload your PDFs to a Volume (see Step 2 below).

## Step 1 — Install the library

In [ ]:
%pip install -q sparkocr-vlm
dbutils.library.restartPython()

## Step 2 — Set your OpenRouter API key

Best practice: store the key in a Databricks secret scope:
```
databricks secrets create-scope sparkocr
databricks secrets put-secret sparkocr openrouter --string-value sk-or-v1-...
```
For a quick demo you can paste the key directly (never commit it).

In [ ]:
import os

try:
    os.environ['OPENROUTER_API_KEY'] = dbutils.secrets.get(scope='sparkocr', key='openrouter')
    print('Key loaded from secret scope.')
except Exception:
    # Fallback: paste key directly for ad-hoc runs
    os.environ['OPENROUTER_API_KEY'] = 'sk-or-v1-PASTE_YOUR_KEY_HERE'
    print('Using hardcoded key — OK for demos, not for production.')

## Step 3 — Create a Volume and upload PDFs

In the Databricks UI:
1. **Catalog** → **workspace** → **default** → **Create Volume** → name it `sparkocr_demo`
2. Open the volume and create a folder called `input`
3. Drag-drop your PDFs into `/Volumes/workspace/default/sparkocr_demo/input/`

Or from your laptop with the Databricks CLI:
```bash
databricks fs cp tests/fixtures/synth_invoice.pdf /Volumes/workspace/default/sparkocr_demo/input/
databricks fs cp tests/fixtures/synth_report.pdf  /Volumes/workspace/default/sparkocr_demo/input/
databricks fs cp tests/fixtures/synth_table.pdf   /Volumes/workspace/default/sparkocr_demo/input/
```

In [ ]:
INPUT  = '/Volumes/workspace/default/sparkocr_demo/input/'
OUTPUT = '/Volumes/workspace/default/sparkocr_demo/silver/'

# Verify PDFs are in place
display(dbutils.fs.ls(INPUT))

## Step 4 — Run the OCR pipeline

Uses `nvidia/nemotron-nano-12b-v2-vl:free` — verified working on OpenRouter free tier, $0.00 cost.

In [ ]:
from sparkocr_vlm import OCRPipeline

pipeline = OCRPipeline(
    backend='openrouter',
    model='nvidia/nemotron-nano-12b-v2-vl:free',
    input_path=INPUT,
    output_path=OUTPUT,
    batch_size=2,        # keep low on free tier to avoid rate limits
    max_cost_usd=0.50,   # hard budget cap
)

silver = pipeline.run(spark)
display(silver)

## Step 5 — Query the Delta silver table

In [ ]:
spark.read.format('delta').load(OUTPUT).createOrReplaceTempView('ocr_silver')

In [ ]:
%%sql
SELECT filename, page_num, LEFT(markdown, 200) AS markdown_preview, cost_usd, error
FROM ocr_silver
ORDER BY filename, page_num

## Step 6 — Cost summary

In [ ]:
summary = spark.read.format('delta').load(OUTPUT).selectExpr(
    'count(*) as total_pages',
    'sum(prompt_tokens) as total_prompt_tokens',
    'sum(completion_tokens) as total_completion_tokens',
    'round(sum(cost_usd), 6) as total_cost_usd',
    'count(error) as pages_with_errors',
)
display(summary)